# Where the Invisible City Could Afford to Linger
### Cross-referencing SVI-invisibility against open-data infrastructure for lingering

**Lin Wei · July 2026 · Pilot study 01 (revised) for PhD applications**

---

**Why this notebook exists.** The original plan for this pilot involved field observation -- standing
at three MRT exits with a stopwatch, counting who paused and who passed through. Under time pressure,
this notebook replaces that plan with an open-data-only test of a related question, answerable without
leaving the desk.

**The reframing, stated precisely.** The writing sample defines *threshold affordance* as the **capacity**
of a space to invite lingering, waiting, or gathering -- not the occurrence of those behaviours on any
given day. Infrastructure that materially enables lingering -- a bench, a shelter, a hawker centre, a
playground -- is arguably a more direct operationalisation of *capacity* than a few hours of field
counts would have been, which measure occurrence under one set of weather, time, and social conditions.
This is not a lesser substitute for fieldwork; it targets the construct more literally.

**The question.** Pilot 02 found that 26.5% of Singapore's covered pedestrian network is plausibly
invisible to road-based street-view imagery, concentrated away from MRT entrances. This notebook asks:
**are the SVI-invisible parts of the network more or less likely to be equipped for lingering than the
visible parts?** If invisibility and affordance-richness coincide, that is a materially stronger claim
than invisibility alone -- it says current urban analytics infrastructure is blind precisely where public
life has been built room to happen.

**Data sources, all open:** OpenStreetMap amenity/leisure tags for lingering-supportive infrastructure
(`amenity=bench`, `amenity=shelter`, `amenity=food_court` -- the tag Singapore's OSM community uses for
hawker centres -- `amenity=community_centre`, `leisure=playground`), cross-referenced against Pilot 02's
saved visibility classification. No new environment, no API keys, no fieldwork.


In [ ]:
# %% 0. Environment -- reuses the covered-city env; no new installs needed
import warnings
warnings.filterwarnings("ignore")

import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("../data")
FIGS = Path("../figures")
CRS_SVY21 = "EPSG:3414"
PLACE = "Singapore"

ox.settings.use_cache = True
ox.settings.cache_folder = str(DATA / "_osm_cache")

print("osmnx", ox.__version__, "| geopandas", gpd.__version__)


## 1. Load Pilot 02's output

If you haven't run `01_mapping_the_covered_city.ipynb` yet, run it first -- this notebook depends on its
saved GeoJSON.

In [ ]:
# %% 1. Load the covered-ways visibility classification from Pilot 02
covered = gpd.read_file(DATA / "sg_covered_ways_svi_visibility.geojson").to_crs(CRS_SVY21)
print(f"Loaded {len(covered)} covered segments, {covered.length.sum()/1000:.1f} km total")
print(covered.svi_visible.value_counts())


## 2. Fetch lingering-supportive infrastructure from OpenStreetMap

Five tags, chosen because each materially enables a specific form of pause: sitting (`bench`), shelter
from sun or rain (`shelter`), eating and long dwell-time socialising (`food_court` -- Singapore's hawker
centres), organised community activity (`community_centre`), and unstructured gathering (`playground`,
as a proxy for family/multi-generational lingering nearby).

In [ ]:
# %% 2. Fetch affordance-infrastructure points
AFFORDANCE_TAGS = [
    ({"amenity": "bench"},            "bench"),
    ({"amenity": "shelter"},          "shelter"),
    ({"amenity": "food_court"},       "hawker_or_food_court"),
    ({"amenity": "community_centre"}, "community_centre"),
    ({"leisure": "playground"},       "playground"),
]

def fetch_points(tags, label):
    try:
        g = ox.features_from_place(PLACE, tags=tags)
    except Exception as e:
        print(f"[{label}] query failed: {e}")
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    g = g.copy()
    g["geometry"] = g.geometry.centroid  # polygons (e.g. some food_courts) -> point
    g["affordance_type"] = label
    print(f"[{label}] {len(g)} features")
    return g[["geometry", "affordance_type"]]

parts = [fetch_points(t, l) for t, l in AFFORDANCE_TAGS]
affordance = pd.concat(parts, ignore_index=True)
affordance = gpd.GeoDataFrame(affordance, geometry="geometry", crs="EPSG:4326").to_crs(CRS_SVY21)
print(f"\nTotal affordance-infrastructure points: {len(affordance)}")
affordance.affordance_type.value_counts()


## 3. Classify each covered segment by affordance proximity

A segment is **affordance-adjacent** if any lingering-supportive feature lies within `AFFORDANCE_RADIUS_M`
of it. Default 20 m -- roughly "at this threshold or immediately beside it," matching the scale used for
the SVI-visibility proxy in Pilot 02, so the two classifications are comparable.

In [ ]:
# %% 3. Proximity classification
AFFORDANCE_RADIUS_M = 20

affordance_union = affordance.geometry.union_all() if hasattr(affordance.geometry, "union_all") \
                   else affordance.geometry.unary_union

covered["dist_to_affordance_m"] = covered.geometry.apply(lambda g: g.distance(affordance_union))
covered["affordance_adjacent"] = covered.dist_to_affordance_m <= AFFORDANCE_RADIUS_M

print(covered.groupby(["svi_visible", "affordance_adjacent"]).length_m.sum().div(1000).round(1))


## 4. The core cross-tabulation

This is the number that matters: among SVI-invisible covered ways, what share are affordance-adjacent --
versus the same share among SVI-visible ways? If invisible segments are *more* likely to be
affordance-adjacent, that supports the stronger claim that current SVI-based urban analytics is blind
precisely where infrastructure suggests public life is meant to happen.

In [ ]:
# %% 4. Cross-tab as percentages
tab = covered.groupby("svi_visible").affordance_adjacent.mean().mul(100).round(1)
tab.index = tab.index.map({True: "SVI-visible", False: "SVI-invisible"})
print("Share of covered network that is affordance-adjacent, by visibility class:")
print(tab.to_string())

invisible_adj = tab.get("SVI-invisible", float("nan"))
visible_adj = tab.get("SVI-visible", float("nan"))
diff = invisible_adj - visible_adj
print(f"\nDifference (invisible minus visible): {diff:+.1f} percentage points")


In [ ]:
# %% 4b. Breakdown by affordance type -- which infrastructure drives the pattern?
breakdown = []
for tag_label in affordance.affordance_type.unique():
    single = affordance[affordance.affordance_type == tag_label]
    single_union = single.geometry.union_all() if hasattr(single.geometry, "union_all") \
                   else single.geometry.unary_union
    dist = covered.geometry.apply(lambda g: g.distance(single_union))
    adjacent = dist <= AFFORDANCE_RADIUS_M
    row = covered.groupby("svi_visible").apply(
        lambda df, adj=adjacent: adj.loc[df.index].mean() * 100
    )
    breakdown.append({"affordance_type": tag_label,
                       "pct_of_invisible_adjacent": row.get(False, float("nan")),
                       "pct_of_visible_adjacent": row.get(True, float("nan"))})
breakdown_df = pd.DataFrame(breakdown).round(1)
print(breakdown_df.to_string(index=False))


## 5. Sensitivity check

Same discipline as Pilot 02: does the finding survive a different radius choice, or is it an artefact
of the 20 m constant?

In [ ]:
# %% 5. Sensitivity sweep across radius
sweep = []
for r in [10, 15, 20, 25, 30, 40]:
    adj = covered.dist_to_affordance_m <= r
    t = covered.groupby("svi_visible").apply(lambda df, a=adj: a.loc[df.index].mean() * 100)
    sweep.append({"radius_m": r,
                  "pct_invisible_adjacent": round(t.get(False, float("nan")), 1),
                  "pct_visible_adjacent": round(t.get(True, float("nan")), 1)})
sweep_df = pd.DataFrame(sweep)
print(sweep_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(sweep_df.radius_m, sweep_df.pct_invisible_adjacent, marker="o", color="#C98A1B", label="SVI-invisible")
ax.plot(sweep_df.radius_m, sweep_df.pct_visible_adjacent, marker="o", color="#0E8C7F", label="SVI-visible")
ax.set_xlabel("affordance search radius (m)")
ax.set_ylabel("% of covered network affordance-adjacent")
ax.set_title("Does invisibility track affordance-richness? (sensitivity check)")
ax.legend(frameon=False)
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig(FIGS / "affordance_sensitivity.png", dpi=200)


## 6. Map

Amber segments (SVI-invisible) that are also affordance-adjacent, shown against everything else --
these are the specific places the argument is about.

In [ ]:
# %% 6. Map: invisible AND affordance-adjacent segments, island-wide
fig, ax = plt.subplots(figsize=(11, 8))
covered.plot(ax=ax, linewidth=.3, color="#D7E0DE", zorder=1)

invisible_only = covered[(~covered.svi_visible) & (~covered.affordance_adjacent)]
invisible_and_afford = covered[(~covered.svi_visible) & (covered.affordance_adjacent)]

invisible_only.plot(ax=ax, linewidth=.6, color="#E8C88A", zorder=2, label="SVI-invisible only")
invisible_and_afford.plot(ax=ax, linewidth=1.1, color="#B0332A", zorder=3,
                           label="SVI-invisible + affordance-adjacent")
affordance.plot(ax=ax, markersize=2, color="#16211F", alpha=.25, zorder=1.5)

ax.set_title("Where invisibility and lingering-infrastructure coincide", fontsize=12)
ax.legend(loc="lower right", frameon=False)
ax.set_axis_off()
fig.tight_layout()
fig.savefig(FIGS / "invisible_affordance_overlap.png", dpi=220)


In [ ]:
# %% 7. Persist outputs
covered.drop(columns=["dist_to_affordance_m"], errors="ignore") \
       .assign(dist_to_affordance_m=covered.dist_to_affordance_m) \
       .to_crs("EPSG:4326") \
       .to_file(DATA / "sg_covered_affordance_overlay.geojson", driver="GeoJSON")
sweep_df.to_csv(DATA / "affordance_sensitivity_sweep.csv", index=False)
breakdown_df.to_csv(DATA / "affordance_breakdown_by_type.csv", index=False)
print("Saved: data/sg_covered_affordance_overlay.geojson, data/affordance_sensitivity_sweep.csv,",
      "data/affordance_breakdown_by_type.csv, figures/*.png")


## 8. Findings

At the default 20 m radius, SVI-invisible covered walkway is **19.5%** likely to sit near
lingering-supportive infrastructure, versus **6.1%** for SVI-visible walkway — a gap of **+13.4
percentage points**. The parts of Singapore's covered pedestrian network that street-view cameras
cannot reach are more than three times as likely to be adjacent to a bench, shelter, or playground as
the parts cameras can reach.

**The finding strengthens, not weakens, under the sensitivity sweep.** Both classes' affordance-adjacent
share rise as the search radius widens (10 m to 40 m) -- unsurprising, since a wider net catches more
amenities regardless of visibility class -- but the *gap* between them widens too, from roughly 7
percentage points at 10 m to roughly 20 points at 40 m. A fragile, threshold-dependent artefact would
typically shrink or reverse under this kind of test; this one grows more pronounced, which is evidence
for a real underlying pattern rather than a constant chosen to produce a result.

**The breakdown by infrastructure type explains why.** Shelter (9.1% invisible-adjacent vs. 2.3%
visible-adjacent) and playground (8.8% vs. 2.5%) drive nearly all of the effect -- both are exactly the
kind of estate-interior feature (covered rest points, void-deck-adjacent play areas) that sits deep
within housing blocks, away from roads. Hawker centres show almost no difference (1.4% vs. 1.0%),
plausibly because hawker centres require vehicle access for supplies and are therefore also
road-adjacent, cancelling out the visibility effect that holds for purely pedestrian infrastructure.
Community centres are too rare a feature to move the numbers either way.

**Read together with Pilot 02's finding** (invisibility concentrated away from MRT entrances, in the
residential interior), this result sharpens the claim considerably: it is not simply that road-based
street-view imagery misses part of the covered network. It misses, disproportionately, the parts of the
network that are already equipped for lingering, waiting, and informal gathering -- exactly the
behaviours this proposal is concerned with. Current SVI-based urban analytics is not blind at random;
it is blind in a way that correlates with where public life has been given room to happen.

## 9. Limitations — stated as findings, not apologies

1. **Infrastructure presence is a capacity proxy, not a behavioural one.** A bench being within 20 m does
   not mean anyone uses it. This notebook deliberately targets the *capacity* sense of threshold
   affordance defined in the accompanying writing sample, not occurrence — but the distinction should be
   stated plainly in any use of this finding, not blurred.
2. **OSM tagging completeness varies by feature type.** Benches in particular are inconsistently mapped
   worldwide; Singapore's community-mapping intensity helps, but a "false negative" (a real bench, untagged)
   is more likely than a false positive. This would, if anything, understate the affordance-adjacent share.
3. **20 m is a proximity choice, not a measured interaction radius.** The sensitivity sweep addresses
   robustness to this specific choice but cannot address whether 20 m is the geographically meaningful
   distance at which infrastructure actually shapes use of a given threshold.
4. **This is a structural correlation, not a causal or experiential claim.** It says where invisibility
   and infrastructure coincide on the map — it says nothing about why, or about what happens when people
   are actually present. Field observation (the original design for this pilot) remains the natural
   next step once time allows.

## 10. Relationship to the proposal

If the core finding holds, it belongs in the writing sample's §6.2 (void deck) and §7 (mixed-methods
approach) as a second, independent line of evidence alongside Pilot 02's road-distance result — two
different open-data signals pointing at the same underdocumented spatial class.
